In [ ]:
import sys
sys.path.append("src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import quantum_algorithms as qa

In [ ]:
f_const = qa.deutsch_jozsa.f_constant_1
f_bal = qa.deutsch_jozsa.f_balanced_parity

n_values = range(1, 8)
theta_values = np.linspace(0, np.pi, 181)

axes = {
    "X": (1, 0, 0),
    "Y": (0, 1, 0),
    "Z": (0, 0, 1),
}

error_positions = {
    "E1_before_H": qa.deutsch_jozsa.deutsch_jozsa_error1,
    "E2_after_first_H": qa.deutsch_jozsa.deutsch_jozsa_error2,
    "E3_after_oracle": qa.deutsch_jozsa.deutsch_jozsa_error3,
    "E4_after_final_H": qa.deutsch_jozsa.deutsch_jozsa_error4,
}

label_map = {
    "no_error": "No Error",
    "E1_before_H": r"$\mathcal{E}_1$ before first $H$",
    "E2_after_first_H": r"$\mathcal{E}_2$ after first $H$",
    "E3_after_oracle": r"$\mathcal{E}_3$ after oracle",
    "E4_after_final_H": r"$\mathcal{E}_4$ after final $H$",
}

shots = 1024
rng = np.random.default_rng(12345)
results = []



# 2. Helper function

def compute_success_probabilities(state_const, state_bal, n, shots, rng):
    samples_const = qa.deutsch_jozsa.sample_measurements_input(
        state_const, n, shots, rng
    )
    samples_bal = qa.deutsch_jozsa.sample_measurements_input(
        state_bal, n, shots, rng
    )

    P0_constant = samples_const[0] / shots
    P0_balanced = samples_bal[0] / shots

    success_constant = P0_constant
    success_balanced = 1 - P0_balanced
    average_success = (success_constant + success_balanced) / 2

    return P0_constant, P0_balanced, success_constant, success_balanced, average_success



# 3. Run full experiment

for n in n_values:
    print(f"Running n = {n}")

    state_const_ref = qa.deutsch_jozsa.deutsch_jozsa(n, f_const)
    state_bal_ref = qa.deutsch_jozsa.deutsch_jozsa(n, f_bal)

    P0_c, P0_b, S_c, S_b, S_avg = compute_success_probabilities(
        state_const_ref, state_bal_ref, n, shots, rng
    )

    results.append({
        "n": n,
        "theta_rad": np.nan,
        "theta_deg": np.nan,
        "axis": "none",
        "target_qubit": np.nan,
        "error_position": "no_error",
        "P0_constant": P0_c,
        "P0_balanced": P0_b,
        "success_constant": S_c,
        "success_balanced": S_b,
        "average_success": S_avg,
        "shots": shots,
    })

    for theta in theta_values:
        theta_deg = np.degrees(theta)

        for axis_name, axis in axes.items():
            for target_qubit in range(n):

                for error_name, error_function in error_positions.items():

                    state_const = error_function(
                        n, f_const, theta, target_qubit, axis
                    )
                    state_bal = error_function(
                        n, f_bal, theta, target_qubit, axis
                    )

                    P0_c, P0_b, S_c, S_b, S_avg = compute_success_probabilities(
                        state_const, state_bal, n, shots, rng
                    )

                    results.append({
                        "n": n,
                        "theta_rad": theta,
                        "theta_deg": theta_deg,
                        "axis": axis_name,
                        "target_qubit": target_qubit,
                        "error_position": error_name,
                        "P0_constant": P0_c,
                        "P0_balanced": P0_b,
                        "success_constant": S_c,
                        "success_balanced": S_b,
                        "average_success": S_avg,
                        "shots": shots,
                    })



# 4. Create and save dataset

df = pd.DataFrame(results)

csv_name = "dja_full_master_thesis_results_n1_to_9_shots_1024.csv"
df.to_csv(csv_name, index=False)

df.to_csv(
    "n1_to_12_shots_1024.csv.gz",
    index=False,
    compression="gzip"
)

print("Saved:", csv_name)
print(df.shape)
df.head()

In [ ]:
def plot_average_success(df, n_plot, axis_plot, target_plot):

    plot_df = df[
        (df["n"] == n_plot) &
        (df["axis"] == axis_plot) &
        (df["target_qubit"] == target_plot)
    ]

    plt.figure(figsize=(9, 6))

    no_error = df[
        (df["n"] == n_plot) &
        (df["error_position"] == "no_error")
    ]

    if len(no_error) > 0:
        plt.axhline(
            no_error["average_success"].iloc[0],
            linestyle="--",
            linewidth=2,
            label=label_map["no_error"]
        )

    for error_pos in error_positions.keys():
        subset = plot_df[plot_df["error_position"] == error_pos]

        plt.plot(
            subset["theta_deg"],
            subset["average_success"],
            linewidth=2,
            label=label_map[error_pos]
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)", fontsize=12)
    plt.ylabel("Average success probability", fontsize=12)
    plt.title(
        f"Deutsch–Jozsa Algorithm under Rotation Errors\n"
        f"n={n_plot}, axis={axis_plot}, target qubit={target_plot}",
        fontsize=13
    )
    plt.xlim(0, 180)
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_average_success(df, n_plot=1, axis_plot="X", target_plot=0)
plot_average_success(df, n_plot=4, axis_plot="X", target_plot=2)
plot_average_success(df, n_plot=6, axis_plot="X", target_plot=5)
plot_average_success(df, n_plot=8, axis_plot="X", target_plot=7)

In [ ]:
plot_average_success(df, n_plot=1, axis_plot="Y", target_plot=0)
plot_average_success(df, n_plot=4, axis_plot="Y", target_plot=2)
plot_average_success(df, n_plot=6, axis_plot="Y", target_plot=5)
plot_average_success(df, n_plot=8, axis_plot="Y", target_plot=7)

In [ ]:
plot_average_success(df, n_plot=1, axis_plot="Z", target_plot=0)
plot_average_success(df, n_plot=4, axis_plot="Z", target_plot=2)
plot_average_success(df, n_plot=6, axis_plot="Z", target_plot=5)
plot_average_success(df, n_plot=8, axis_plot="Z", target_plot=7)